In [4]:
!pip install sacrebleu
!pip install transformers
!pip install accelerate
!pip install evaluate

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
import os
direc = "/dss/dsshome1/09/go34jos2"
src = os.path.join(direc,"dev.de-hsb.de")
tgt =os.path.join(direc,"dev.de-hsb.hsb")
from transformers import AutoTokenizer, AutoModelForCausalLM
import re
import torch
from tqdm import tqdm
import evaluate
if os.path.exists(src) and os.path.exists(tgt):
  with open(src,  encoding= 'utf-8' ) as f:
    val_s = [line.strip() for line in f.readlines() if line.strip()]
  with open(tgt ,encoding = 'utf-8') as f:
    val_t = [line.strip() for line in f.readlines() if line.strip()]
  model_name = "Qwen/Qwen3-0.6B"
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForCausalLM.from_pretrained(
      model_name,
      dtype=torch.bfloat16,
      device_map="auto",
  )
  if tokenizer.pad_token is None:
    tokenizer.pad_token= tokenizer.eos_token
  tokenizer.padding_side ="left"
  if tokenizer.chat_template is None:
    tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
  def extract_translation(text):
    match = re.search(r"<hsb>\s*(.*?)\s*</hsb>",text, re.DOTALL)
    if match:
      return  match.group(1).strip()
    return text.strip()
  predictions = []
  batch_size =16
  #nrc template
  instruction_text = "Translate the following German text to Upper Sorbian.\nPut it in this format <hsb> Upper Sorbian translation </hsb>."
  for i in tqdm(range(0,len(val_s), batch_size)):
    batch_src =val_s[i:i+batch_size]
    batch_prompts = []
    for src in batch_src:
      messages = [
          {"role": "system", "content": instruction_text},
          {"role": "user", "content": f"<deu> {src} </deu>"}
      ]
      text= tokenizer.apply_chat_template(messages,tokenize=False, add_generation_prompt=True)
      batch_prompts.append(text)
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.no_grad():
      generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
      )
    generated_ids =[out[len(inp):] for inp, out in  zip(inputs.input_ids, generated_ids)]
    raw_preds =tokenizer.batch_decode(generated_ids,skip_special_tokens=True)
    clean_preds =[ extract_translation(p) for p in raw_preds]
    predictions.extend(clean_preds)
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")
bleu_result =sacrebleu.compute(predictions=predictions, references=[[r] for r in val_t])
chrf_result = chrf.compute(predictions=predictions, references=[[r] for r in val_t], word_order=2)
print(f"SacreBLEU: {bleu_result['score']:.2f}")
print(f"chrF++:    {chrf_result['score']:.2f}")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 250/250 [2:12:26<00:00, 31.79s/it]  


SacreBLEU: 0.00
chrF++:    1.31
